# Batch Model Monitoring Demo - Setup

Run once to set up the environment:
1. Database, schema, warehouse
2. Tables: CUSTOMERS, SUBSCRIPTION_EVENTS, SCORING_DATA, BASELINE_DATA
3. Seed realistic customer and event data

In [ ]:
!pip install --upgrade "snowflake-ml-python>=1.7.1" snowflake-connector-python

In [ ]:
%%sql -r df_setup
CREATE WAREHOUSE IF NOT EXISTS ML_DEMO_WH WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
CREATE DATABASE IF NOT EXISTS ML_DEMOS;
CREATE SCHEMA IF NOT EXISTS ML_DEMOS.BATCH_MONITORING;
USE DATABASE ML_DEMOS;
USE SCHEMA BATCH_MONITORING;
USE WAREHOUSE ML_DEMO_WH;

In [ ]:
%%sql -r df_tables
CREATE OR REPLACE TABLE CUSTOMERS (
    customer_id VARCHAR(20) NOT NULL,
    signup_date TIMESTAMP_NTZ,
    country VARCHAR(50),
    plan_type VARCHAR(20),
    age INT,
    tenure_months INT,
    monthly_charges FLOAT,
    total_charges FLOAT,
    contract_type VARCHAR(20),
    num_support_tickets INT,
    days_since_last_login INT,
    avg_monthly_usage_hours FLOAT,
    PRIMARY KEY (customer_id)
);

CREATE OR REPLACE TABLE SUBSCRIPTION_EVENTS (
    event_id VARCHAR(36) NOT NULL,
    customer_id VARCHAR(20) NOT NULL,
    event_type VARCHAR(30) NOT NULL,
    event_ts TIMESTAMP_NTZ NOT NULL,
    PRIMARY KEY (event_id)
);

CREATE OR REPLACE TABLE SCORING_DATA (
    id VARCHAR(36) NOT NULL,
    customer_id VARCHAR(20) NOT NULL,
    prediction_ts TIMESTAMP_NTZ NOT NULL,
    tenure_months INT,
    monthly_charges FLOAT,
    total_charges FLOAT,
    num_support_tickets INT,
    days_since_last_login INT,
    avg_monthly_usage_hours FLOAT,
    age INT,
    contract_type VARCHAR(20),
    plan_type VARCHAR(20),
    prediction_score FLOAT,
    PRIMARY KEY (id)
);

CREATE OR REPLACE TABLE BASELINE_DATA (
    id VARCHAR(36) NOT NULL,
    customer_id VARCHAR(20) NOT NULL,
    prediction_ts TIMESTAMP_NTZ NOT NULL,
    tenure_months INT,
    monthly_charges FLOAT,
    total_charges FLOAT,
    num_support_tickets INT,
    days_since_last_login INT,
    avg_monthly_usage_hours FLOAT,
    age INT,
    contract_type VARCHAR(20),
    plan_type VARCHAR(20),
    prediction_score FLOAT,
    PRIMARY KEY (id)
);

In [ ]:
import random
import uuid
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.use_schema("ML_DEMOS.BATCH_MONITORING")

random.seed(42)

COUNTRIES = ["US", "US", "US", "CA", "UK", "DE", "AU", "FR", "JP", "BR"]
PLAN_TYPES = ["BASIC", "BASIC", "BASIC", "PREMIUM", "PREMIUM", "ENTERPRISE"]
CONTRACT_TYPES = ["MONTH_TO_MONTH", "MONTH_TO_MONTH", "MONTH_TO_MONTH", "ONE_YEAR", "ONE_YEAR", "TWO_YEAR"]

NUM_CUSTOMERS = 2000
customers = []
base_date = datetime(2024, 1, 1)

for i in range(NUM_CUSTOMERS):
    cid = f"C{i+1:05d}"
    tenure = random.randint(1, 72)
    signup = base_date - timedelta(days=tenure * 30)
    plan = random.choice(PLAN_TYPES)
    contract = random.choice(CONTRACT_TYPES)

    base_charge = {"BASIC": 29.99, "PREMIUM": 59.99, "ENTERPRISE": 99.99}[plan]
    monthly = round(base_charge * random.uniform(0.8, 1.3), 2)

    customers.append({
        "CUSTOMER_ID": cid,
        "SIGNUP_DATE": signup.strftime("%Y-%m-%d %H:%M:%S"),
        "COUNTRY": random.choice(COUNTRIES),
        "PLAN_TYPE": plan,
        "AGE": random.randint(18, 70),
        "TENURE_MONTHS": tenure,
        "MONTHLY_CHARGES": monthly,
        "TOTAL_CHARGES": round(monthly * tenure * random.uniform(0.9, 1.0), 2),
        "CONTRACT_TYPE": contract,
        "NUM_SUPPORT_TICKETS": random.choices([0,0,0,1,1,2,3,5,8], k=1)[0],
        "DAYS_SINCE_LAST_LOGIN": random.choices(
            [0,1,1,2,3,5,7,14,30,60], k=1
        )[0],
        "AVG_MONTHLY_USAGE_HOURS": round(max(0.5, random.gauss(45, 20)), 1),
    })

df = pd.DataFrame(customers)
sp_df = session.create_dataframe(df)
sp_df.write.mode("overwrite").save_as_table("CUSTOMERS")
print(f"Seeded {NUM_CUSTOMERS} customers")
session.table("CUSTOMERS").show(5)

In [ ]:
# Generate subscription events over 6 months
EVENT_TYPES = ["login", "login", "login", "feature_use", "feature_use",
               "support_ticket", "plan_change", "payment", "payment"]

events = []
start_date = datetime(2024, 7, 1)
end_date = datetime(2024, 12, 31)
total_days = (end_date - start_date).days

for cust in customers:
    num_events = random.randint(10, 50)
    for _ in range(num_events):
        event_ts = start_date + timedelta(
            days=random.randint(0, total_days),
            hours=random.randint(6, 22),
            minutes=random.randint(0, 59)
        )
        events.append({
            "EVENT_ID": str(uuid.uuid4()),
            "CUSTOMER_ID": cust["CUSTOMER_ID"],
            "EVENT_TYPE": random.choice(EVENT_TYPES),
            "EVENT_TS": event_ts.strftime("%Y-%m-%d %H:%M:%S"),
        })

events_df = pd.DataFrame(events)
sp_events = session.create_dataframe(events_df)
sp_events.write.mode("overwrite").save_as_table("SUBSCRIPTION_EVENTS")
print(f"Seeded {len(events):,} subscription events")
session.table("SUBSCRIPTION_EVENTS").show(5)

In [ ]:
%%sql -r df_verify
SELECT 'CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM CUSTOMERS
UNION ALL
SELECT 'SUBSCRIPTION_EVENTS', COUNT(*) FROM SUBSCRIPTION_EVENTS
UNION ALL
SELECT 'SCORING_DATA', COUNT(*) FROM SCORING_DATA
UNION ALL
SELECT 'BASELINE_DATA', COUNT(*) FROM BASELINE_DATA;